# Adding Monastic Opposition Variables

Joins `monastic_opposition.csv` to the parish shapefile using the same 20 km proximity / 10 km IDW logic applied to the main gentlemen variables in `jn_05`.

For each monastery flagged in the opposition CSV, a point is placed at its lat/lon. Parish-level dummies and IDW-weighted exposure scores are then computed for three groups:

- **Crown interference = 1** (strong): 9 houses where the Crown installed or removed the head pre-1536.
- **Crown interference = 0.5** (mild): 5 houses with documented but softer Crown pressure.
- **Any opposition = 1**: 10 houses with leadership or rank-and-file opposition to the Henrician Reformation before the Pilgrimage of Grace.

**Input:** `Data/Raw/CSV/monastic_opposition.csv`, `Data/Processed/northParishFlows.shp`
**Output:** `Data/Processed/northParishFlows.shp` (updated in place)

## Loading

In [1]:
print("Loading packages and data...")
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# Paths relative to project root
PROJECT_ROOT = Path.cwd().parent
RAW  = PROJECT_ROOT / 'Data' / 'Raw'
PROCESSED = PROJECT_ROOT / 'Data' / 'Processed'

# Input paths
OPP_CSV    = RAW / 'CSV' / 'monastic_opposition.csv'
PARISH_SHP = PROCESSED / 'northParishFlows.shp'

# Output path
OUTPUT_SHP = PROCESSED / 'northParishFlows.shp'

opp_df = pd.read_csv(OPP_CSV)
parish_flows = gpd.read_file(PARISH_SHP)

print(f"Loaded {len(opp_df)} monasteries from monastic_opposition.csv")
print(f"Loaded {len(parish_flows)} parishes")
print("Monastery counts by group:")
print(f"  crown_interference == 1.0: {int((opp_df['crown_interference'] == 1.0).sum())}")
print(f"  crown_interference == 0.5: {int((opp_df['crown_interference'] == 0.5).sum())}")
print(f"  any_opposition     == 1  : {int((opp_df['any_opposition'] == 1).sum())}")

Loading packages and data...
Loaded 218 monasteries from monastic_opposition.csv
Loaded 1755 parishes
Monastery counts by group:
  crown_interference == 1.0: 9
  crown_interference == 0.5: 5
  any_opposition     == 1  : 10


## Convert to GeoDataFrame and reproject to BNG

Coordinates in the CSV are WGS84 (EPSG:4326); the parish shapefile uses British National Grid (EPSG:27700). Reproject so that distance arithmetic is in metres.

In [2]:
opp_gdf = gpd.GeoDataFrame(
    opp_df,
    geometry=gpd.points_from_xy(opp_df['lon'], opp_df['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:27700')

print(f"GeoDataFrame CRS: {opp_gdf.crs}")
print(f"Bounding box (BNG metres): {opp_gdf.total_bounds}")

GeoDataFrame CRS: EPSG:27700
Bounding box (BNG metres): [296884.55200483 306471.01250365 548740.55460119 641774.41341881]


## Binary variables for 20 km proximity

For each opposition group, a binary (0/1) indicator is created: does the parish centroid fall within 20 km of at least one monastery in that group?

| Output column | Definition |
|---|---|
| `mo_ci1`     | Crown interference = 1 (strong) |
| `mo_ci05`    | Crown interference = 0.5 (mild) |
| `mo_anyop`  | Any opposition = 1 |

In [3]:
BUFFER_M = 20_000

# Each definition is a boolean mask over opp_gdf rows
group_masks = {
    'mo_ci1':    opp_gdf['crown_interference'] == 1.0,
    'mo_ci05':   opp_gdf['crown_interference'] == 0.5,
    'mo_anyop': opp_gdf['any_opposition'] == 1,
}

for out_col, mask in group_masks.items():
    subset = opp_gdf[mask]
    if len(subset) == 0:
        parish_flows[out_col] = 0
        print(f"{out_col}: 0 monasteries -> 0 parishes flagged")
        continue
    buffer = subset.geometry.buffer(BUFFER_M).union_all()
    parish_flows[out_col] = parish_flows.geometry.centroid.within(buffer).astype(int)
    print(f"{out_col}: {len(subset)} monasteries -> {int(parish_flows[out_col].sum())} parishes flagged")

mo_ci1: 9 monasteries -> 393 parishes flagged
mo_ci05: 5 monasteries -> 197 parishes flagged
mo_anyop: 10 monasteries -> 348 parishes flagged


## Inverse-distance-weighted (IDW) versions

For each parish centroid and opposition group, sum a weight across all monasteries in the group:

```
w(d) = 1            if d <= 10 km
w(d) = 10 / d_km    if d >  10 km
```

So a monastery 20 km away contributes 0.5, 30 km -> 0.33, etc. The IDW score is the sum of weights across all monasteries in the group. Matches the formula used for the main gentlemen IDW variables in `jn_05`.

| Output column | Definition |
|---|---|
| `mo_ci1_w`     | Crown interference = 1 (strong) IDW |
| `mo_ci05_w`    | Crown interference = 0.5 (mild) IDW |
| `mo_anyop_w`  | Any opposition = 1 IDW |

In [4]:
FLAT_RADIUS_M = 10_000  # 10 km flat zone; beyond this weight = FLAT_RADIUS_M / d_m

# Parish centroid coordinates as numpy arrays (BNG metres)
centroids = parish_flows.geometry.centroid
cx = centroids.x.values
cy = centroids.y.values

idw_group_masks = {
    'mo_ci1_w':    opp_gdf['crown_interference'] == 1.0,
    'mo_ci05_w':   opp_gdf['crown_interference'] == 0.5,
    'mo_anyop_w': opp_gdf['any_opposition'] == 1,
}

for out_col, mask in idw_group_masks.items():
    subset = opp_gdf[mask]
    if len(subset) == 0:
        parish_flows[out_col] = 0.0
        print(f"{out_col}: 0 monasteries -> all zeros")
        continue

    mx = subset.geometry.x.values
    my = subset.geometry.y.values

    # Pairwise distances: shape (n_parishes, n_monasteries)
    dist_m = np.sqrt(
        (cx[:, np.newaxis] - mx[np.newaxis, :]) ** 2
        + (cy[:, np.newaxis] - my[np.newaxis, :]) ** 2
    )
    # Weight: 1 inside flat zone, FLAT_RADIUS_M / d outside (avoids div-by-zero at d=0)
    weights = np.where(dist_m <= FLAT_RADIUS_M, 1.0, FLAT_RADIUS_M / dist_m)

    parish_flows[out_col] = weights.sum(axis=1)
    col = parish_flows[out_col]
    print(f"{out_col}: {len(subset)} monasteries  "
          f"min={col.min():.3f}  mean={col.mean():.3f}  max={col.max():.3f}")

mo_ci1_w: 9 monasteries  min=0.521  mean=1.085  max=2.189
mo_ci05_w: 5 monasteries  min=0.245  mean=0.657  max=2.238
mo_anyop_w: 10 monasteries  min=0.547  mean=1.341  max=3.261


## Save updated shapefile

In [5]:
parish_flows.to_file(OUTPUT_SHP)

print(f"Updated shapefile saved to {OUTPUT_SHP}")
print("New binary columns:")
for c in ['mo_ci1', 'mo_ci05', 'mo_anyop']:
    print(f"  - {c}: {int(parish_flows[c].sum())} parishes = 1")
print("New IDW columns (min / mean / max):")
for c in ['mo_ci1_w', 'mo_ci05_w', 'mo_anyop_w']:
    v = parish_flows[c]
    print(f"  - {c}: {v.min():.3f} / {v.mean():.3f} / {v.max():.3f}")

Updated shapefile saved to c:\Users\nicho\My Drive\20_Projects\NRP---New-Rebellion-Paper\Data\Processed\northParishFlows.shp
New binary columns:
  - mo_ci1: 393 parishes = 1
  - mo_ci05: 197 parishes = 1
  - mo_anyop: 348 parishes = 1
New IDW columns (min / mean / max):
  - mo_ci1_w: 0.521 / 1.085 / 2.189
  - mo_ci05_w: 0.245 / 0.657 / 2.238
  - mo_anyop_w: 0.547 / 1.341 / 3.261


c:\Users\nicho\My Drive\20_Projects\NRP---New-Rebellion-Paper\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Creating a 256th field, but some DBF readers might only support 255 fields
  ogr_write(
